# Week 2 — Text Processing and Tokenization

How a string becomes a sequence of integers. We start with whitespace splitting, then progressively rebuild every modern subword algorithm: Byte-Pair Encoding (BPE), WordPiece, and the Unigram LM used in SentencePiece.

## Learning Objectives

- Explain the trade-offs between character-level, word-level, and subword tokenization.
- Implement BPE from scratch and train it on a real corpus.
- Implement WordPiece (likelihood-based merge criterion) and the Unigram LM via EM.
- Benchmark a from-scratch tokenizer against `tokenizers` on speed and compression ratio.

## Required Reading

- Sennrich, R., Haddow, B., & Birch, A. (2016). *Neural Machine Translation of Rare Words with Subword Units*.
- Kudo, T. (2018). *Subword Regularization*.
- Schuster, M., & Nakajima, K. (2012). *Japanese and Korean Voice Search*.

In [ ]:
import sys, re, unicodedata
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

np.random.seed(0)

## 1. Normalization and pre-tokenization

Every tokenizer begins with two steps that are deceptively important:

- **Unicode normalization.** The character *é* can be encoded as one codepoint (NFC) or as *e* + combining accent (NFD). NFC is the standard choice.
- **Pre-tokenization.** A coarse split that BPE then refines. The most common scheme: split on whitespace and isolate punctuation.

In [ ]:
def normalize(text, lower=True, form='NFC'):
    text = unicodedata.normalize(form, text)
    return text.lower() if lower else text

def pre_tokenize(text):
    # Splits whitespace-separated chunks, then peels off leading/trailing punctuation.
    return re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE)

sample = "Don't tokenize naïvely — it's harder than it looks."
print(normalize(sample))
print(pre_tokenize(normalize(sample)))

## 2. Byte-Pair Encoding from scratch

BPE was originally a compression algorithm (Gage, 1994), adapted to NLP by Sennrich et al. (2016).

**Training algorithm.**

1. Initialize the vocabulary with all characters in the corpus.
2. Represent each word as a sequence of characters terminated by a special end-of-word marker `</w>`.
3. Count all adjacent symbol pairs across the corpus.
4. Merge the most frequent pair into a new symbol; add it to the vocabulary.
5. Repeat until the vocabulary reaches the desired size $V$.

**Encoding a new word.** Apply the learned merges in the order they were learned.

In [ ]:
class BPETokenizer:
    def __init__(self):
        self.merges = []                  # ordered list of (a, b) pair merges
        self.vocab = set()

    def _word_to_symbols(self, word):
        return list(word) + ['</w>']

    def _get_pairs(self, splits):
        # splits: dict[word_tuple] -> count
        pairs = Counter()
        for symbols, count in splits.items():
            for i in range(len(symbols) - 1):
                pairs[(symbols[i], symbols[i+1])] += count
        return pairs

    def _merge_pair(self, splits, pair):
        new_splits = {}
        a, b = pair
        for symbols, count in splits.items():
            new = []
            i = 0
            while i < len(symbols):
                if i < len(symbols) - 1 and symbols[i] == a and symbols[i+1] == b:
                    new.append(a + b)
                    i += 2
                else:
                    new.append(symbols[i])
                    i += 1
            new_splits[tuple(new)] = count
        return new_splits

    def train(self, corpus, num_merges=200, verbose=False):
        word_counts = Counter(pre_tokenize(normalize(corpus)))
        splits = {tuple(self._word_to_symbols(w)): c for w, c in word_counts.items()}
        self.vocab = set(s for symbols in splits for s in symbols)

        for step in range(num_merges):
            pairs = self._get_pairs(splits)
            if not pairs:
                break
            best = pairs.most_common(1)[0][0]
            self.merges.append(best)
            self.vocab.add(best[0] + best[1])
            splits = self._merge_pair(splits, best)
            if verbose and step < 5:
                print(f"step {step:3d}: merge {best} -> {best[0]+best[1]!r}")
        return self

    def encode_word(self, word):
        symbols = self._word_to_symbols(word)
        for a, b in self.merges:
            i = 0
            while i < len(symbols) - 1:
                if symbols[i] == a and symbols[i+1] == b:
                    symbols = symbols[:i] + [a + b] + symbols[i+2:]
                else:
                    i += 1
        return symbols

    def encode(self, text):
        out = []
        for w in pre_tokenize(normalize(text)):
            out.extend(self.encode_word(w))
        return out

# Train on a tiny corpus to see what merges form.
toy_corpus = ("low low low low low "
              "lower lower newer newer newer newer newer newer "
              "wider wider wider widest widest")
bpe = BPETokenizer().train(toy_corpus, num_merges=10, verbose=True)
print(f"\nVocabulary size: {len(bpe.vocab)}")
print(f"Encoding 'lowest': {bpe.encode('lowest')}")
print(f"Encoding 'newest': {bpe.encode('newest')}")

In [ ]:
# Train on a more realistic corpus and look at compression.
big_corpus = (open(__file__).read() if False else
              "the quick brown fox jumps over the lazy dog. " * 200 +
              "tokenization is the first step in any natural language pipeline. " * 200 +
              "subword units help with out of vocabulary words. " * 200 +
              "agglutinative morphology benefits enormously from subword segmentation. " * 200)

for V in [50, 200, 500, 1000]:
    bpe = BPETokenizer().train(big_corpus, num_merges=V)
    tokens = bpe.encode("agglutinative tokenizers are extraordinarily useful.")
    print(f"merges={V:4d} | vocab={len(bpe.vocab):4d} | "
          f"encoded into {len(tokens):2d} tokens: {tokens}")

## 3. WordPiece — likelihood-based merging

WordPiece (Schuster & Nakajima, 2012; used by BERT) replaces BPE's frequency criterion with a likelihood criterion. Define a candidate merge of pair $(a, b)$ and let $\text{count}(\cdot)$ denote corpus counts. The score is:

$$\text{score}(a, b) = \frac{\text{count}(ab)}{\text{count}(a) \cdot \text{count}(b)}.$$

Intuition: merge pairs whose joint frequency far exceeds what independence would predict — exactly pointwise mutual information without the log. The merge that maximizes this score is performed.

In [ ]:
class WordPieceTokenizer:
    def __init__(self):
        self.merges = []
        self.vocab = set()

    def _get_pair_and_token_counts(self, splits):
        pair_counts = Counter()
        token_counts = Counter()
        for symbols, count in splits.items():
            for s in symbols:
                token_counts[s] += count
            for i in range(len(symbols) - 1):
                pair_counts[(symbols[i], symbols[i+1])] += count
        return pair_counts, token_counts

    def _merge_pair(self, splits, pair):
        a, b = pair
        new_splits = {}
        for symbols, count in splits.items():
            new = []
            i = 0
            while i < len(symbols):
                if i < len(symbols) - 1 and symbols[i] == a and symbols[i+1] == b:
                    new.append(a + b); i += 2
                else:
                    new.append(symbols[i]); i += 1
            new_splits[tuple(new)] = count
        return new_splits

    def train(self, corpus, num_merges=200):
        word_counts = Counter(pre_tokenize(normalize(corpus)))
        # WordPiece convention: continuations are prefixed with '##'.
        splits = {}
        for w, c in word_counts.items():
            symbols = [w[0]] + ['##' + ch for ch in w[1:]]
            splits[tuple(symbols)] = c
        self.vocab = set(s for symbols in splits for s in symbols)

        for _ in range(num_merges):
            pair_counts, tok_counts = self._get_pair_and_token_counts(splits)
            if not pair_counts:
                break
            # score = count(ab) / (count(a) * count(b))
            scores = {p: c / (tok_counts[p[0]] * tok_counts[p[1]])
                      for p, c in pair_counts.items()}
            best = max(scores, key=scores.get)
            merged = best[0] + (best[1][2:] if best[1].startswith('##') else best[1])
            self.merges.append((best, merged))
            self.vocab.add(merged)
            splits = self._merge_pair(splits, best)
        return self

wp = WordPieceTokenizer().train(big_corpus, num_merges=200)
print(f"WordPiece vocabulary size: {len(wp.vocab)}")
print(f"First 10 merges:")
for pair, merged in wp.merges[:10]:
    print(f"  {pair} -> {merged!r}")

## 4. Unigram LM (SentencePiece)

Kudo (2018) frames tokenization as a probabilistic model: each token in vocabulary $V$ has a probability $p(t)$, and the best segmentation of a word $w$ is

$$\arg\max_{s \in S(w)} \prod_{t \in s} p(t),$$

where $S(w)$ is the set of valid segmentations. Training:

1. Start with a large seed vocabulary (e.g. all substrings up to length $L$).
2. Estimate $p(t)$ by EM: for each word, compute expected counts via the forward algorithm (or just Viterbi for a simplified variant).
3. Prune the lowest-utility tokens to shrink $|V|$.
4. Repeat until $|V|$ reaches the target.

The implementation below is the **Viterbi-only** simplification, which still illustrates the core ideas.

In [ ]:
class UnigramTokenizer:
    def __init__(self, vocab_size=100):
        self.vocab_size = vocab_size
        self.log_probs = {}

    def _seed_vocab(self, corpus, max_len=6):
        # All substrings up to length max_len, weighted by frequency.
        counts = Counter()
        for w in pre_tokenize(normalize(corpus)):
            for i in range(len(w)):
                for j in range(i+1, min(i+max_len, len(w))+1):
                    counts[w[i:j]] += 1
        return counts

    def _viterbi_segment(self, word):
        # Best segmentation under current log_probs. Returns (tokens, score).
        n = len(word)
        best = [(-1e18, None)] * (n + 1)
        best[0] = (0.0, None)
        for i in range(1, n + 1):
            for j in range(max(0, i - 16), i):
                sub = word[j:i]
                if sub in self.log_probs:
                    score = best[j][0] + self.log_probs[sub]
                    if score > best[i][0]:
                        best[i] = (score, (j, sub))
        # Backtrack
        if best[n][1] is None:
            return list(word), -1e18  # fall back to chars
        tokens, i = [], n
        while i > 0:
            j, sub = best[i][1]
            tokens.append(sub); i = j
        return tokens[::-1], best[n][0]

    def train(self, corpus, num_iters=5, shrink_factor=0.75):
        seed = self._seed_vocab(corpus)
        total = sum(seed.values())
        self.log_probs = {t: np.log(c / total) for t, c in seed.items()}
        word_counts = Counter(pre_tokenize(normalize(corpus)))

        for it in range(num_iters):
            # E-step: collect expected counts via Viterbi segmentation.
            new_counts = Counter()
            for w, c in word_counts.items():
                tokens, _ = self._viterbi_segment(w)
                for t in tokens:
                    new_counts[t] += c
            total = sum(new_counts.values()) or 1
            self.log_probs = {t: np.log(c / total) for t, c in new_counts.items() if c > 0}

            # Shrink: drop lowest-prob tokens (but keep all single chars).
            target = max(self.vocab_size, int(len(self.log_probs) * shrink_factor))
            if len(self.log_probs) > self.vocab_size:
                kept = sorted(self.log_probs.items(), key=lambda kv: kv[1], reverse=True)
                must_keep = {t for t in self.log_probs if len(t) == 1}
                pruned = {t for t, _ in kept[:target]} | must_keep
                self.log_probs = {t: lp for t, lp in self.log_probs.items() if t in pruned}
        return self

    def encode_word(self, w):
        return self._viterbi_segment(w)[0]

uni = UnigramTokenizer(vocab_size=120).train(big_corpus, num_iters=4)
print(f"Final vocab size: {len(uni.log_probs)}")
print(f"Encoding 'tokenization': {uni.encode_word('tokenization')}")
print(f"Encoding 'agglutinative': {uni.encode_word('agglutinative')}")

## 5. Comparing the three on Turkish

Turkish is agglutinative — morphologically rich. The same English noun in English/Turkish:

> *house* / *ev* → *in my houses* / *evlerimde* (ev-ler-im-de = house-PLURAL-1SG.POSS-LOC)

A good tokenizer should find morpheme-like boundaries.

In [ ]:
TR_CORPUS = ("ev evler evim evimde evlerim evlerimde " * 100 +
             "kitap kitabı kitabım kitaplar kitaplarım kitaplarımda " * 100 +
             "gel geldim geldin geliyorum geleceğim gelecektim " * 100).lower()

bpe_tr = BPETokenizer().train(TR_CORPUS, num_merges=80)
wp_tr  = WordPieceTokenizer().train(TR_CORPUS, num_merges=80)
uni_tr = UnigramTokenizer(vocab_size=60).train(TR_CORPUS, num_iters=4)

test_words = ['evlerimde', 'kitaplarım', 'geleceğim']
print(f"{'word':<15} {'BPE':<35} {'WordPiece':<35} {'Unigram'}")
for w in test_words:
    print(f"{w:<15} {str(bpe_tr.encode_word(w)):<35} "
          f"{str([s.replace('##','') for s in [w[0]] + ['##'+c for c in w[1:]]])[:33]:<35} "
          f"{uni_tr.encode_word(w)}")

## 6. Benchmark: from-scratch vs. Hugging Face `tokenizers`

If `tokenizers` is installed, this cell will benchmark speed and compression. Otherwise it gracefully skips.

In [ ]:
import time
try:
    from tokenizers import Tokenizer
    from tokenizers.models import BPE
    from tokenizers.trainers import BpeTrainer
    from tokenizers.pre_tokenizers import Whitespace

    hf = Tokenizer(BPE(unk_token='[UNK]'))
    hf.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(vocab_size=500, special_tokens=['[UNK]'])
    hf.train_from_iterator([big_corpus], trainer)

    text = big_corpus[:5000]
    t0 = time.perf_counter(); ours = BPETokenizer().train(big_corpus, num_merges=500).encode(text); t1 = time.perf_counter()
    t2 = time.perf_counter(); theirs = hf.encode(text).tokens; t3 = time.perf_counter()
    print(f"Ours   : {len(ours):5d} tokens, {(t1-t0)*1000:6.1f} ms")
    print(f"HF     : {len(theirs):5d} tokens, {(t3-t2)*1000:6.1f} ms")
except ImportError:
    print("`tokenizers` not installed — skipping benchmark.")
    print("Install with: pip install tokenizers")

## 7. Exercises

1. **Compression curve.** Train BPE with $V \in \{500, 2000, 8000, 32000\}$ on a 10 MB text. Plot average tokens-per-sentence against $V$. At what $V$ do diminishing returns set in?
2. **Byte-level BPE.** GPT-2 operates on bytes rather than characters — this guarantees no `[UNK]` token ever. Implement a byte-level variant and verify it can encode arbitrary text including emoji and code.
3. **Turkish morpheme recovery.** Hand-annotate the morpheme boundaries of 20 Turkish words. Measure how often each tokenizer recovers them. Which wins, and why?
4. **Unigram LM with EM.** The implementation above uses Viterbi (hard EM). Replace it with the forward algorithm (soft EM) and compare convergence.

---

## Next Week

Week 3 — Classical Text Representations. Bag-of-words, TF-IDF, n-gram language models, and LSA, all from NumPy.